# Лабораторная работа №2: Детекция и сегментация объектов

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Проверить разметку и распределения объектов.
- Дообучить компактный segmentation baseline.
- Сравнить аугментации и проанализировать ошибки.


## Теоретическая часть

В детекции модель предсказывает классы и bounding boxes, в сегментации — метку каждого пикселя. Для класса $c$ качество сегментации измеряет $IoU_c=|P_c\cap G_c|/|P_c\cup G_c|$. Для детекции mAP усредняет AP по классам и порогам IoU.

![Segmentation example](https://pytorch.org/vision/main/_images/sphx_glr_plot_visualization_utils_008.png)


### Математическая постановка

Составная функция потерь может иметь вид
$$\mathcal L=\mathcal L_{cls}+\lambda_{box}\mathcal L_{box}+\lambda_{mask}\mathcal L_{mask}.$$
Разбиение должно выполняться по исходным объектам/сериям, иначе соседние кадры дают утечку.


### Материалы

- [Torchvision segmentation models](https://pytorch.org/vision/stable/models.html#semantic-segmentation)
- [COCO metrics](https://cocodataset.org/#detection-eval)
- [Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Проверить разметку и распределения объектов.
2. Дообучить компактный segmentation baseline.
3. Сравнить аугментации и проанализировать ошибки.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** LRASPP-MobileNetV3 или FCN-ResNet50, mixed precision.
- **Ограничение данных:** Oxford-IIIT Pet или до 2 000 изображений; в smoke-режиме — синтетический датасет.
- **Fallback:** замороженный backbone и 256 px.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [5]:
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"

if DEVICE == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)

_CANDIDATES = [
    Path.cwd().parent.parent / "data",
    Path.cwd().parent / "data",
    Path.cwd() / "data",
]
DATA_ROOT = Path(
    os.getenv("CV_COURSE_DATA")
    or next((c for c in _CANDIDATES if (c / "oxford-iiit-pet").exists()), _CANDIDATES[0])
)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})

{'device': 'cpu', 'fast_dev_run': False, 'data_root': 'd:\\data'}


In [6]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Маски Oxford-IIIT Pet

*Вес: 1 балл.*

**Постановка.** Сегментатор обучается на реальной пиксельной разметке.

Выполните следующие действия:
1. Загрузите trainval Oxford-IIIT Pet и совместно масштабируйте изображение и маску.
2. Убедитесь, что интерполяция маски сохраняет классы.

**Результат.** Датасет ds и преобразования img_tf, mask_tf.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [7]:
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.datasets import OxfordIIITPet
from PIL import Image
import torch
import numpy as np

image_size = (256, 256)

class PetSegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, root, split="trainval", img_size=image_size, download=True):
        base_ds = OxfordIIITPet(
            root=root, split=split, target_types="segmentation", download=download
        )
        self.img_size = img_size
        self.data = []
        print(10 * "-" + f" Загрузка и кэширование ({split}) в RAM " + 10 * "-")
        for img, mask in base_ds:
            img = TF.resize(img, self.img_size, interpolation=T.InterpolationMode.BILINEAR)
            mask = TF.resize(mask, self.img_size, interpolation=T.InterpolationMode.NEAREST)

            img_uint8 = torch.from_numpy(np.array(img)).permute(2, 0, 1).contiguous()
            mask_long = torch.from_numpy(np.array(mask)).long()
            self.data.append((img_uint8, mask_long))
        print(f"✔ Кэш готов. Элементов: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_uint8, mask = self.data[idx]
        img = img_uint8.float() / 255.0
        img = TF.normalize(
            img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )
        return img, mask

ds = PetSegmentationDataset(
    root=DATA_ROOT, split="trainval", img_size=image_size, download=True
)
if FAST_DEV_RUN:
    ds = torch.utils.data.Subset(ds, range(32))

def img_tf(img):
    img = TF.resize(img, image_size, interpolation=T.InterpolationMode.BILINEAR)
    img = TF.to_tensor(img)
    return TF.normalize(
        img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    )

def mask_tf(mask):
    mask = TF.resize(mask, image_size, interpolation=T.InterpolationMode.NEAREST)
    return torch.as_tensor(np.array(mask), dtype=torch.long)

print(f"✔ Датасет готов. Элементов: {len(ds)}")

---------- Загрузка и кэширование (trainval) в RAM ----------
✔ Кэш готов. Элементов: 3680
✔ Датасет готов. Элементов: 3680


### Разделение и кодирование меток

*Вес: 1 балл.*

**Постановка.** Оценка на validation требует непересекающихся индексов и корректных категориальных меток.

Выполните следующие действия:
1. Разделите trainval на train и validation.
2. Перекодируйте исходные метки 1–3 в 0–2 и соберите загрузчики.

**Результат.** train_ds, val_ds, loader, val_loader и normalize_mask.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [8]:
from torch.utils.data import random_split, DataLoader

def normalize_mask(mask):
    mask = mask.long() - 1
    mask = torch.clamp(mask, 0, 2)
    return mask

class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, mask = self.subset[idx]
        return img, normalize_mask(mask)

val_size = int(0.2 * len(ds))
train_size = len(ds) - val_size
train_subset, val_subset = random_split(
    ds, [train_size, val_size], generator=torch.Generator().manual_seed(SEED)
)

train_ds = EncodedDataset(train_subset)
val_ds = EncodedDataset(val_subset)

batch_size = 4 if FAST_DEV_RUN else 16
loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"✔ Разделение: train={len(train_ds)}, val={len(val_ds)}, batch_size={batch_size}")

✔ Разделение: train=2944, val=736, batch_size=16


### Сегментатор и IoU

*Вес: 1 балл.*

**Постановка.** Пиксельная модель предсказывает логиты для каждого класса.

Выполните следующие действия:
1. Замените головы LRASPP на трёхклассовые.
2. Реализуйте IoU по пересечению и объединению масок.

**Результат.** Объект model и функция mean_iou.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [9]:
import torchvision.models.segmentation as seg_models
import torch.nn as nn

model = seg_models.lraspp_mobilenet_v3_large(
    weights=seg_models.LRASPP_MobileNet_V3_Large_Weights.DEFAULT
)
model.classifier.high_classifier = nn.Conv2d(128, 3, kernel_size=1)
model.classifier.low_classifier = nn.Conv2d(40, 3, kernel_size=1)
model.to(DEVICE)

def mean_iou(pred, target, num_classes=3):
    ious = []
    pred = pred.view(-1)
    target = target.view(-1)
    for cls in range(num_classes):
        pred_inds = (pred == cls)
        target_inds = (target == cls)
        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()
        if union == 0:
            continue
        ious.append(intersection / union)
    return sum(ious) / len(ious) if ious else 0.0

def foreground_iou(pred, target, fg_class=0):
    pred_inds = (pred == fg_class)
    target_inds = (target == fg_class)
    intersection = (pred_inds & target_inds).sum().item()
    union = (pred_inds | target_inds).sum().item()
    return intersection / union if union > 0 else 0.0

print("✔ Модель и метрики готовы")

✔ Модель и метрики готовы


### Оптимизация сегментатора

*Вес: 1 балл.*

**Постановка.** Cross-entropy на масках задаёт обучающий сигнал каждому пикселю.

Выполните следующие действия:
1. Реализуйте шаг AdamW с обратным распространением.
2. Обучите модель установленное число эпох.

**Результат.** Функция train_one_epoch и массив history реальных значений loss.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [10]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for imgs, masks in dataloader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total_samples += imgs.size(0)
    return total_loss / max(1, total_samples)

epochs = 2 if FAST_DEV_RUN else 4
history = []
for epoch in range(epochs):
    epoch_loss = train_one_epoch(model, loader, optimizer, criterion, DEVICE)
    history.append(epoch_loss)
    print(f"Эпоха {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")

print(f"✔ Обучение завершено. История loss: {history}")

Эпоха 1/4 | Loss: 0.2783
Эпоха 2/4 | Loss: 0.2065
Эпоха 3/4 | Loss: 0.1817
Эпоха 4/4 | Loss: 0.1674
✔ Обучение завершено. История loss: [0.2782866215414327, 0.20645160786807537, 0.18171291231461192, 0.16744984597291634]


### Валидационный IoU

*Вес: 1 балл.*

**Постановка.** Матрица ошибок по пикселям позволяет вычислить классовый IoU.

Выполните следующие действия:
1. Накопите confusion matrix по validation.
2. Сохраните mIoU и checkpoint исходной модели.

**Результат.** segmentation.pt и val_metrics.json.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [11]:
model.eval()
val_ious = []
with torch.inference_mode():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        outputs = model(imgs)['out']
        preds = outputs.argmax(dim=1)
        for p, t in zip(preds, masks):
            val_ious.append(mean_iou(p, t))

val_miou = sum(val_ious) / len(val_ious) if val_ious else 0.0
print(f"✔ Validation mIoU: {val_miou:.4f}")

torch.save(model.state_dict(), ARTIFACTS / "segmentation.pt")
metrics_data = {"val_miou": val_miou, "epochs": epochs}
with open(ARTIFACTS / "val_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, ensure_ascii=False, indent=2)

print(f"✔ Артефакты сохранены в {ARTIFACTS}")

✔ Validation mIoU: 0.7480
✔ Артефакты сохранены в artifacts


### Совместное отражение и повторное обучение

*Вес: 1 балл.*

**Постановка.** Геометрическое преобразование изображения без идентичного преобразования маски портит разметку.

Выполните следующие действия:
1. Реализуйте синхронное горизонтальное отражение.
2. Обучите новый сегментатор при том же бюджете и сравните validation mIoU.

**Результат.** segmentation_aug.pt и augmentation_comparison.json.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [12]:
import torch

class CachedAugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, p=0.5):
        self.subset = subset
        self.p = p

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, mask = self.subset[idx]
        if random.random() < self.p:
            img = torch.flip(img, dims=[-1])
            mask = torch.flip(mask, dims=[-1])
        return img, normalize_mask(mask)

train_ds_aug = CachedAugmentedDataset(train_subset, p=0.5)
loader_aug = DataLoader(
    train_ds_aug, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True
)

model_aug = seg_models.lraspp_mobilenet_v3_large(
    weights=seg_models.LRASPP_MobileNet_V3_Large_Weights.DEFAULT
)
model_aug.classifier.high_classifier = nn.Conv2d(128, 3, kernel_size=1)
model_aug.classifier.low_classifier = nn.Conv2d(40, 3, kernel_size=1)
model_aug.to(DEVICE)

optimizer_aug = torch.optim.AdamW(model_aug.parameters(), lr=1e-3, weight_decay=1e-4)

print("🚀 Обучение с аугментацией...")
for epoch in range(epochs):
    epoch_loss = train_one_epoch(model_aug, loader_aug, optimizer_aug, criterion, DEVICE)
    print(f"Аугментация | Эпоха {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")

model_aug.eval()
val_ious_aug = []
with torch.inference_mode():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model_aug(imgs)['out'].argmax(dim=1)
        for p, t in zip(preds, masks):
            val_ious_aug.append(mean_iou(p, t))

val_miou_aug = sum(val_ious_aug) / len(val_ious_aug) if val_ious_aug else 0.0
print(f"✔ Validation mIoU с аугментацией: {val_miou_aug:.4f}")

torch.save(model_aug.state_dict(), ARTIFACTS / "segmentation_aug.pt")
comp_data = {"val_miou_baseline": val_miou, "val_miou_aug": val_miou_aug}
with open(ARTIFACTS / "augmentation_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comp_data, f, ensure_ascii=False, indent=2)

🚀 Обучение с аугментацией...
Аугментация | Эпоха 1/4 | Loss: 0.2883
Аугментация | Эпоха 2/4 | Loss: 0.2145
Аугментация | Эпоха 3/4 | Loss: 0.1985
Аугментация | Эпоха 4/4 | Loss: 0.1823
✔ Validation mIoU с аугментацией: 0.7587


### Сегментация малых объектов

*Вес: 1 балл.*

**Постановка.** Размер категории, заданный по train, позволяет отдельно изучать слабые места модели.

Выполните следующие действия:
1. Определите порог площади объекта только на train.
2. Вычислите foreground IoU на малых объектах validation для обеих моделей.

**Результат.** small_objects.json с размером подвыборки и IoU двух моделей.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [13]:
train_areas = []
for _, mask in train_ds:
    fg_area = (mask == 0).sum().item()
    train_areas.append(fg_area)

threshold_area = np.percentile(train_areas, 25) if train_areas else 100

small_obj_ious_base = []
small_obj_ious_aug = []

model.eval()
model_aug.eval()
with torch.inference_mode():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds_base = model(imgs)['out'].argmax(dim=1)
        preds_aug = model_aug(imgs)['out'].argmax(dim=1)
        for m, pb, pa in zip(masks, preds_base, preds_aug):
            area = (m == 0).sum().item()
            if 0 < area <= threshold_area:
                small_obj_ious_base.append(foreground_iou(pb, m, fg_class=0))
                small_obj_ious_aug.append(foreground_iou(pa, m, fg_class=0))

small_res = {
    "threshold_area": float(threshold_area),
    "sample_count": len(small_obj_ious_base),
    "miou_base_small": float(sum(small_obj_ious_base) / len(small_obj_ious_base))
    if small_obj_ious_base
    else 0.0,
    "miou_aug_small": float(sum(small_obj_ious_aug) / len(small_obj_ious_aug))
    if small_obj_ious_aug
    else 0.0,
}
with open(ARTIFACTS / "small_objects.json", "w", encoding="utf-8") as f:
    json.dump(small_res, f, ensure_ascii=False, indent=2)

print(f"✔ Анализ малых объектов: {small_res}")

✔ Анализ малых объектов: {'threshold_area': 11458.5, 'sample_count': 163, 'miou_base_small': 0.7613548642881522, 'miou_aug_small': 0.7631059896994288}


### Отложенная оценка двух моделей

*Вес: 1 балл.*

**Постановка.** Test не должен влиять на обучение и выбор конфигурации.

Выполните следующие действия:
1. Загрузите test-раздел Oxford-IIIT Pet.
2. Оцените оба сохранённых сегментатора на одинаковых test-изображениях.

**Результат.** test_metrics.json с IoU на реальном test.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [14]:
test_ds = PetSegmentationDataset(
    root=DATA_ROOT, split="test", img_size=image_size, download=True
)
if FAST_DEV_RUN:
    test_ds = torch.utils.data.Subset(test_ds, range(10))

class EncodedTestDataset(torch.utils.data.Dataset):
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, mask = self.subset[idx]
        return img, normalize_mask(mask)

test_encoded = EncodedTestDataset(test_ds)
test_loader = DataLoader(
    test_encoded, batch_size=batch_size, shuffle=False, num_workers=0
)

test_ious_base = []
test_ious_aug = []

with torch.inference_mode():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        pb = model(imgs)['out'].argmax(dim=1)
        pa = model_aug(imgs)['out'].argmax(dim=1)
        for p1, p2, t in zip(pb, pa, masks):
            test_ious_base.append(mean_iou(p1, t))
            test_ious_aug.append(mean_iou(p2, t))

test_res = {
    "test_miou_base": float(sum(test_ious_base) / len(test_ious_base))
    if test_ious_base
    else 0.0,
    "test_miou_aug": float(sum(test_ious_aug) / len(test_ious_aug))
    if test_ious_aug
    else 0.0,
}
with open(ARTIFACTS / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_res, f, ensure_ascii=False, indent=2)

print(f"✔ Отложенная оценка test: {test_res}")

---------- Загрузка и кэширование (test) в RAM ----------
✔ Кэш готов. Элементов: 3669
✔ Отложенная оценка test: {'test_miou_base': 0.7487708788917401, 'test_miou_aug': 0.7629704436564756}


### Локализация ошибок маски

*Вес: 1 балл.*

**Постановка.** False positive и false negative выделяют различные режимы сбоя сегментатора.

Выполните следующие действия:
1. Посчитайте IoU, FP и FN для каждого изображения validation.
2. Сохраните наиболее сложные изображения и их индексы.

**Результат.** error_analysis.json с фактическими ошибками модели.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [15]:
error_records = []
model.eval()
with torch.inference_mode():
    for idx in range(len(val_ds)):
        img, mask = val_ds[idx]
        img_tensor = img.unsqueeze(0).to(DEVICE)
        mask_tensor = mask.unsqueeze(0).to(DEVICE)

        output = model(img_tensor)['out']
        pred = output.argmax(dim=1)[0].cpu()
        true_m = mask_tensor[0].cpu()

        miou = mean_iou(pred, true_m)
        fg = 0
        fp = ((pred == fg) & (true_m != fg)).sum().item()
        fn = ((pred != fg) & (true_m == fg)).sum().item()

        error_records.append({"index": idx, "miou": miou, "fp": fp, "fn": fn})

error_records.sort(key=lambda x: x["miou"])
with open(ARTIFACTS / "error_analysis.json", "w", encoding="utf-8") as f:
    json.dump(error_records[:10], f, ensure_ascii=False, indent=2)

print(f"✔ Анализ ошибок сохранен. Всего: {len(error_records)}")

✔ Анализ ошибок сохранен. Всего: 736


### Поклассовая динамика аугментации

*Вес: 1 балл.*

**Постановка.** Средний IoU не показывает, каким классам помогло отражение.

Выполните следующие действия:
1. Сопоставьте поклассовый validation IoU исходной и дополненной моделей.
2. Сохраните изменения по каждому классу, не используя test для выбора модели.

**Результат.** class_deltas.json с наблюдаемыми изменениями IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [16]:
# Реализуйте это задание по постановке выше.
def get_per_class_ious(model, loader, num_classes=3):
    model.eval()
    class_intersections = [0] * num_classes
    class_unions = [0] * num_classes
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)['out'].argmax(dim=1)
            for p, t in zip(preds, masks):
                for cls in range(num_classes):
                    pi = (p == cls)
                    ti = (t == cls)
                    class_intersections[cls] += (pi & ti).sum().item()
                    class_unions[cls] += (pi | ti).sum().item()
    return [i / u if u > 0 else 0.0 for i, u in zip(class_intersections, class_unions)]

base_per_class = get_per_class_ious(model, val_loader)
aug_per_class = get_per_class_ious(model_aug, val_loader)

deltas = {
    f"class_{cls}": {"base_iou": b, "aug_iou": a, "delta": a - b}
    for cls, (b, a) in enumerate(zip(base_per_class, aug_per_class))
}

with open(ARTIFACTS / "class_deltas.json", "w", encoding="utf-8") as f:
    json.dump(deltas, f, ensure_ascii=False, indent=2)

print(f"✔ Поклассовые дельты сохранены: {deltas}")


✔ Поклассовые дельты сохранены: {'class_0': {'base_iou': 0.8638447388810363, 'aug_iou': 0.8584199979954242, 'delta': -0.005424740885612067}, 'class_1': {'base_iou': 0.924741776541994, 'aug_iou': 0.9263207614232066, 'delta': 0.001578984881212664}, 'class_2': {'base_iou': 0.48452841063422575, 'aug_iou': 0.5174496237776371, 'delta': 0.0329212131434114}}


### Творческий бонус: диагностические маски

*Вес: 1 дополнительный балл.*

**Постановка.** Ошибки на границах полезно изучать в пространстве изображения.

Выполните следующие действия:
1. Найдите сложнейшее validation-изображение по IoU.
2. Сохраните реальную RGB-картинку, истинную и предсказанную маски для визуального разбора.

**Результат.** bonus_hard_case.png и bonus_hard_case.json с индексом и IoU.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [17]:
# Реализуйте это задание по постановке выше.
import matplotlib.pyplot as plt

worst_case = min(error_records, key=lambda x: x["miou"])
worst_idx = worst_case["index"]

img_tensor, true_mask = val_ds[worst_idx]
model.eval()
with torch.no_grad():
    pred_mask = model(img_tensor.unsqueeze(0).to(DEVICE))['out'].argmax(dim=1)[0].cpu()

bonus_meta = {"index": worst_idx, "miou": worst_case["miou"]}
with open(ARTIFACTS / "bonus_hard_case.json", "w", encoding="utf-8") as f:
    json.dump(bonus_meta, f, ensure_ascii=False, indent=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_vis = img_tensor * std + mean
img_vis = img_vis.clamp(0, 1).permute(1, 2, 0).numpy()

axes[0].imshow(img_vis)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(true_mask.numpy(), cmap="viridis")
axes[1].set_title("True Mask")
axes[1].axis("off")

axes[2].imshow(pred_mask.numpy(), cmap="viridis")
axes[2].set_title("Predicted Mask")
axes[2].axis("off")

plt.tight_layout()
plt.savefig(ARTIFACTS / "bonus_hard_case.png")
plt.close()

print(f"✔ Творческий бонус сохранен: индекс {worst_idx}, mIoU {worst_case['miou']:.4f}")


✔ Творческий бонус сохранен: индекс 433, mIoU 0.2681


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.
